In [1]:
import mlflow
import dagshub

C:\Users\KIIT0001\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
mlflow.set_tracking_uri("https://dagshub.com/Aayush10671/mental-health-score-predictor.mlflow")

In [5]:
import dagshub
dagshub.init(repo_owner='Aayush10671', repo_name='mental-health-score-predictor', mlflow=True)

import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

Initialized MLflow to track repo "Aayush10671/mental-health-score-predictor"

Repository Aayush10671/mental-health-score-predictor initialized!

🏃 View run resilient-wasp-461 at: https://dagshub.com/Aayush10671/mental-health-score-predictor.mlflow/#/experiments/0/runs/2acb987311a24ed6aa14172571e29922
🧪 View experiment at: https://dagshub.com/Aayush10671/mental-health-score-predictor.mlflow/#/experiments/0


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer,LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [7]:
mlflow.set_experiment("baseline-models")

2026/08/01 21:38:40 INFO mlflow.tracking.fluent: Experiment with name 'baseline-models' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/5ad9aecf41ab42809bbeec5f9de3097d', creation_time=1785600521373, experiment_id='1', last_update_time=1785600521373, lifecycle_stage='active', name='baseline-models', tags={}, workspace='default'>

In [10]:
df = pd.read_csv('../data/raw/dataset.csv')

In [11]:
num_feature = df.select_dtypes(include = 'number')
len(num_feature.columns)

7

In [12]:
Q1 = num_feature.quantile(0.25)
Q3 = num_feature.quantile(0.75)
IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

outliers = ((num_feature < lower_limit) | (num_feature > upper_limit)).sum()
outliers


Age                         0
Avg_Daily_Usage_Hours       0
Daily_Unlocks               0
Study_Hours                 2
Physical_Activity_Hours    22
Sleep_Hours_Per_Night       0
Mental_Health_Score         0
dtype: int64

In [13]:
df.drop_duplicates(inplace=True)

df['Physical_Activity_Hours'] = df['Physical_Activity_Hours'].clip(lower=0)

In [15]:
top_countries = df['Country'].value_counts().head(11)
def group_countries(country):
    if country in top_countries.index:
        return country
    else:
        return 'Other'

In [16]:
df['Grouped_Country'] = df['Country'].apply(group_countries)

In [17]:
skewed_col = ['Study_Hours']
other_numric_col = ['Age', 'Avg_Daily_Usage_Hours', 'Sleep_Hours_Per_Night', 'Physical_Activity_Hours' , 'Daily_Unlocks']
ordinal_col = ['Stress_Level']

normal_col = ['Gender','Academic_Level','Most_Used_Platform','Grouped_Country' , 'Purpose_Of_Use']  
features_col = skewed_col + other_numric_col + ordinal_col + normal_col

X = df[features_col]
y = df['Mental_Health_Score']

In [18]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder,LabelEncoder,OrdinalEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

skew_pipeline = Pipeline(steps=[('log_transformer', FunctionTransformer(np.log1p, validate=True)), ('scaler', StandardScaler())])
                                
plain_pipeline = Pipeline(steps=[('scaler', StandardScaler())])

ordinal_pipeline = Pipeline(steps=[('ordinal_encoder', OrdinalEncoder(categories=[['Low', 'Medium', 'High' , 'Very High']]))])

nominal_pipeline = Pipeline(steps=[('onehot_encoder', OneHotEncoder(drop='first' , handle_unknown='ignore'))])


preprocessor = ColumnTransformer(transformers=[
    ('skew', skew_pipeline, skewed_col),
    ('plain', plain_pipeline, other_numric_col),
    ('ordinal', ordinal_pipeline, ordinal_col),
    ('nominal', nominal_pipeline, normal_col)
])

In [19]:
with mlflow.start_run():
    mlflow.log_param('model_type', 'RandomForestRegressor')
    mlflow.log_param('n_estimators', 100)
    mlflow.log_param('random_state', 42)
    mlflow.log_param('test_size', 0.3)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    model = Pipeline(steps=[('preprocessor', preprocessor),
                            ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))])

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    mlflow.log_param('model_type', 'RandomForestRegressor')
    mlflow.log_metric('mse', mse)
    mlflow.log_metric('r2', r2)
    mlflow.sklearn.log_model(model, "model")
    import os
    notebook_path = 'exp1.ipynb'
    os.system(f"jupyter nbconvert --to notebook --execute --inplace  {notebook_path}")
    mlflow.log_artifact(notebook_path)

print(f'Mean Squared Error: {mse}')
print(f'R-squared: {r2}')



2026/08/01 21:47:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/01 21:47:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run painted-goat-969 at: https://dagshub.com/Aayush10671/mental-health-score-predictor.mlflow/#/experiments/1/runs/408250607b9b4d9088154ec06a6bd650
🧪 View experiment at: https://dagshub.com/Aayush10671/mental-health-score-predictor.mlflow/#/experiments/1
Mean Squared Error: 0.21270434798166665
R-squared: 0.8788959693804786
